# Kiểm thử Jacobi: bốn hệ phương trình trong ảnh

Chạy notebook từ thư mục `Gau-Jacobi`, sau khi cài các thư viện trong `requirements-lock.txt` ở thư mục gốc.

| Hệ | Kích thước | Sai số cho phép | Nghiệm đối chiếu |
| --- | --- | --- | --- |
| 1 | 4 × 4 | $10^{-5}$ (tự chọn vì ảnh không ghi) | $(1, 2, -1, 1)^T$ |
| 2 | 3 × 3 | $10^{-3}$ | `numpy.linalg.solve(A, b)` |
| 3 | 4 × 4 | $10^{-3}$ | `numpy.linalg.solve(A, b)` |
| 4 | 4 × 4 | $0.5 \times 10^{-3}$ | `numpy.linalg.solve(A, b)` |

Mỗi hệ được kiểm tra với cả hai phiên bản: **v1 chạy đủ 100 bước**, **v2 dừng theo sai số**, tối đa 100 bước. So sánh sai số nghiệm theo chuẩn vô cùng với epsilon tương ứng (`atol=esl`, `rtol=0`). Giữ thêm hai ca kiểm tra việc trả `None` khi không thỏa điều kiện đủ $q<1$: tổng cộng **10 ca kiểm thử**.

Sau phần kiểm thử, in nghiệm cuối của từng hệ cho v1 và v2, với **6 chữ số sau dấu phẩy**.


In [1]:
import ast
from pathlib import Path

import ipytest
import nbformat
import numpy as np
import pytest

ipytest.autoconfig()


# Chỉ đọc định nghĩa hàm từ notebook, không chạy phần ví dụ.
def load_jacobi(filename):
    path = Path(filename)
    if not path.exists():
        raise FileNotFoundError(f"Không tìm thấy file: {filename}")

    notebook = nbformat.read(path, as_version=4)
    namespace = {"np": np}
    for cell in notebook.cells:
        if cell.cell_type != "code":
            continue
        tree = ast.parse(cell.source)
        for node in tree.body:
            if isinstance(node, ast.FunctionDef) and node.name == "Jacobi":
                module = ast.Module(body=[node], type_ignores=[])
                ast.fix_missing_locations(module)
                exec(compile(module, str(path), "exec"), namespace)
                return namespace["Jacobi"]

    raise ValueError(f"Không tìm thấy hàm Jacobi trong {filename}")


jacobi_v1 = load_jacobi("Jacobi_v1.ipynb")
jacobi_v2 = load_jacobi("Jacobi_v2.ipynb")


# Bốn hệ phương trình theo đúng thứ tự trong ảnh.
# Hệ 1 không ghi epsilon: chọn 1e-5 và dùng nghiệm đúng đã cho.
HE_PHUONG_TRINH = [
    pytest.param(
        np.array([
            [10, -1, 2, 0],
            [-1, 11, -1, 3],
            [2, -1, 10, -1],
            [0, 3, -1, 8],
        ], dtype=float),
        np.array([6, 25, -11, 15], dtype=float),
        1e-5,
        np.array([1, 2, -1, 1], dtype=float),
        id="He_1_nghiem_dung",
    ),
    pytest.param(
        np.array([
            [1.02, -0.25, -0.30],
            [-0.41, 1.13, -0.15],
            [-0.25, -0.14, 1.21],
        ], dtype=float),
        np.array([0.515, 1.555, 2.780], dtype=float),
        1e-3,
        None,
        id="He_2_epsilon_1e-3",
    ),
    pytest.param(
        np.array([
            [10.9, 1.2, 2.1, 0.9],
            [1.2, 11.2, 1.5, 2.5],
            [2.1, 1.5, 9.8, 1.3],
            [0.9, 2.5, 1.3, 12.1],
        ], dtype=float),
        np.array([-7, 5.3, 10.3, 24.6], dtype=float),
        1e-3,
        None,
        id="He_3_epsilon_1e-3",
    ),
    pytest.param(
        np.array([
            [4, 0.24, -0.08, 0.16],
            [0.09, 3, -0.155, -0.12],
            [0.04, -0.08, -4, 0.06],
            [0.02, 0.06, 0.04, -10],
        ], dtype=float),
        np.array([8, 9, 20, 1], dtype=float),
        0.5e-3,
        None,
        id="He_4_epsilon_0.5e-3",
    ),
]


# Lưu kết quả của chính các lần chạy kiểm thử để hiển thị bên dưới.
KET_QUA = []


# 4 hệ x 2 phiên bản = 8 ca kiểm thử.
@pytest.mark.parametrize(
    "solver",
    [jacobi_v1, jacobi_v2],
    ids=["Dung_so_buoc_lap", "Dung_theo_sai_so"],
)
@pytest.mark.parametrize("A, b, esl, expected", HE_PHUONG_TRINH)
def test_bon_he_phuong_trinh(solver, A, b, esl, expected, capsys, request):
    max_blap = 100
    if expected is None:
        # Nghiệm tham chiếu độc lập với thuật toán lặp đang kiểm tra.
        expected = np.linalg.solve(A, b)

    result = solver(A, b, max_blap=max_blap, esl=esl)
    output = capsys.readouterr().out

    assert result is not None
    assert np.all(np.isfinite(result))
    np.testing.assert_allclose(result, expected, atol=esl, rtol=0)

    # Kiểm tra hai cách dừng của v1 và v2.
    so_buoc_lap = output.count("KQ lần lặp thứ")
    if solver is jacobi_v1:
        assert so_buoc_lap == max_blap
    else:
        assert 1 <= so_buoc_lap < max_blap
        assert "Hội tụ đạt được sau" in output

    KET_QUA.append({
        "he": int(request.node.callspec.id.split("_")[1]),
        "phien_ban": "v1 — đủ số bước" if solver is jacobi_v1 else "v2 — theo sai số",
        "result": result.copy(),
    })


# Giữ ca biên: thuật toán trả None khi không thỏa điều kiện đủ q < 1.
@pytest.mark.parametrize(
    "solver",
    [jacobi_v1, jacobi_v2],
    ids=["Dung_so_buoc_lap", "Dung_theo_sai_so"],
)
def test_khong_thoa_dieu_kien_q(solver):
    A = np.array([[1, 2], [2, 1]], dtype=float)
    b = np.array([3, 3], dtype=float)

    result = solver(A, b, max_blap=10, esl=1e-6)

    assert result is None


# Tổng cộng 10 ca: 8 ca cho bốn hệ và 2 ca biên.
exit_code = ipytest.run("-q")
assert exit_code == pytest.ExitCode.OK, "Có ca kiểm thử thất bại."

for item in KET_QUA:
    nghiem = "; ".join(f"{value:.6f}" for value in item["result"])
    print(f"Hệ {item['he']} — {item['phien_ban']}")
    print(f"Nghiệm: [{nghiem}]\n")


.

.

.

.

.

.

... (2 outputs left out)

.

.

                                                                                   [100%]

Hệ 1 — v1 — đủ số bước
Nghiệm: [1.000000; 2.000000; -1.000000; 1.000000]

Hệ 1 — v2 — theo sai số
Nghiệm: [1.000001; 1.999999; -0.999999; 0.999999]

Hệ 2 — v1 — đủ số bước
Nghiệm: [2.000000; 2.500000; 3.000000]

Hệ 2 — v2 — theo sai số
Nghiệm: [1.999524; 2.499529; 2.999666]

Hệ 3 — v1 — đủ số bước
Nghiệm: [-1.000000; -0.000000; 1.000000; 2.000000]

Hệ 3 — v2 — theo sai số
Nghiệm: [-1.000120; -0.000134; 0.999857; 1.999880]

Hệ 4 — v1 — đủ số bước
Nghiệm: [1.742262; 2.683426; -5.037754; -0.100566]

Hệ 4 — v2 — theo sai số
Nghiệm: [1.742237; 2.683416; -5.037763; -0.100563]



In [1]:
import ast
from pathlib import Path

import ipytest
import nbformat
import numpy as np
import pytest

ipytest.autoconfig()


# Chỉ đọc định nghĩa hàm từ notebook, không chạy phần ví dụ.
def load_jacobi(filename):
    path = Path(filename)

    if not path.exists():
        raise FileNotFoundError(f"Không tìm thấy file: {filename}")

    notebook = nbformat.read(path, as_version=4)
    namespace = {"np": np}

    for cell in notebook.cells:
        if cell.cell_type != "code":
            continue

        tree = ast.parse(cell.source)

        for node in tree.body:
            if isinstance(node, ast.FunctionDef) and node.name == "jacobi":
                module = ast.Module(body=[node], type_ignores=[])
                ast.fix_missing_locations(module)

                exec(compile(module, str(path), "exec"), namespace)
                return namespace["jacobi"]

    raise ValueError(f"Không tìm thấy hàm jacobi trong {filename}")


# Tải hai phiên bản Jacobi
jacobi_v1 = load_jacobi("Jacobi_v1.ipynb")
jacobi_v2 = load_jacobi("Jacobi_v2.ipynb")


# Bốn hệ phương trình kiểm thử
HE_PHUONG_TRINH = [
    pytest.param(
        np.array([
            [10, -1, 2, 0],
            [-1, 11, -1, 3],
            [2, -1, 10, -1],
            [0, 3, -1, 8],
        ], dtype=float),
        np.array([6, 25, -11, 15], dtype=float),
        1e-5,
        np.array([1, 2, -1, 1], dtype=float),
        id="He_1_nghiem_dung",
    ),
    pytest.param(
        np.array([
            [1.02, -0.25, -0.30],
            [-0.41, 1.13, -0.15],
            [-0.25, -0.14, 1.21],
        ], dtype=float),
        np.array([0.515, 1.555, 2.780], dtype=float),
        1e-3,
        None,
        id="He_2_epsilon_1e-3",
    ),
    pytest.param(
        np.array([
            [10.9, 1.2, 2.1, 0.9],
            [1.2, 11.2, 1.5, 2.5],
            [2.1, 1.5, 9.8, 1.3],
            [0.9, 2.5, 1.3, 12.1],
        ], dtype=float),
        np.array([-7, 5.3, 10.3, 24.6], dtype=float),
        1e-3,
        None,
        id="He_3_epsilon_1e-3",
    ),
    pytest.param(
        np.array([
            [4, 0.24, -0.08, 0.16],
            [0.09, 3, -0.155, -0.12],
            [0.04, -0.08, -4, 0.06],
            [0.02, 0.06, 0.04, -10],
        ], dtype=float),
        np.array([8, 9, 20, 1], dtype=float),
        0.5e-3,
        None,
        id="He_4_epsilon_0.5e-3",
    ),
]


# Lưu kết quả kiểm thử
KET_QUA = []


# 4 hệ x 2 phiên bản = 8 ca kiểm thử
@pytest.mark.parametrize(
    "solver",
    [jacobi_v1, jacobi_v2],
    ids=["Jacobi_v1", "Jacobi_v2"],
)
@pytest.mark.parametrize("A, b, esl, expected", HE_PHUONG_TRINH)
def test_bon_he_phuong_trinh(solver, A, b, esl, expected, capsys, request):

    n_max = 100
    x0 = np.zeros(len(b))

    # Tính nghiệm tham chiếu
    if expected is None:
        expected = np.linalg.solve(A, b)

    # Gọi hàm theo chữ ký mới
    result, k, err = solver(A, b, x0, eps=esl, n_max=n_max)

    output = capsys.readouterr().out

    # Kiểm tra nghiệm
    assert result is not None
    assert np.all(np.isfinite(result))

    np.testing.assert_allclose(
        result,
        expected,
        atol=esl,
        rtol=0
    )

    # Kiểm tra số bước lặp
    so_buoc_lap = output.count("KQ lần lặp thứ")

    assert so_buoc_lap == k
    assert 0 <= k <= n_max

    # Kiểm tra sai số trả về
    assert np.isfinite(err)
    assert err >= 0

    # Kiểm tra điều kiện dừng
    assert err < esl

    # Lưu kết quả
    KET_QUA.append({
        "he": int(request.node.callspec.id.split("_")[1]),
        "phien_ban": (
            "Jacobi v1"
            if solver is jacobi_v1
            else "Jacobi v2"
        ),
        "result": result.copy(),
        "k": k,
        "err": err,
    })


# 2 ca kiểm thử điều kiện q < 1
@pytest.mark.parametrize(
    "solver",
    [jacobi_v1, jacobi_v2],
    ids=["Jacobi_v1", "Jacobi_v2"],
)
def test_khong_thoa_dieu_kien_q(solver):

    A = np.array([
        [1, 2],
        [2, 1]
    ], dtype=float)

    b = np.array([3, 3], dtype=float)
    x0 = np.zeros(len(b))

    # Hàm mới phát sinh ValueError
    with pytest.raises(ValueError):
        solver(A, b, x0, eps=1e-6, n_max=10)


# Chạy 10 ca kiểm thử
exit_code = ipytest.run("-q")

assert exit_code == pytest.ExitCode.OK, "Có ca kiểm thử thất bại."


# Hiển thị kết quả
for item in KET_QUA:

    nghiem = "; ".join(
        f"{value:.6f}" for value in item["result"]
    )

    print(f"Hệ {item['he']} — {item['phien_ban']}")
    print(f"Nghiệm: [{nghiem}]")
    print(f"Số lần lặp: {item['k']}")
    print(f"Sai số: {item['err']:.6f}\n")


..........                                                                                   [100%]
Hệ 1 — Jacobi v1
Nghiệm: [1.000001; 1.999999; -0.999999; 0.999999]
Số lần lặp: 16
Sai số: 0.000005

Hệ 1 — Jacobi v2
Nghiệm: [1.000001; 1.999999; -0.999999; 0.999999]
Số lần lặp: 16
Sai số: 0.000005

Hệ 2 — Jacobi v1
Nghiệm: [1.999524; 2.499529; 2.999666]
Số lần lặp: 11
Sai số: 0.000682

Hệ 2 — Jacobi v2
Nghiệm: [1.999524; 2.499529; 2.999666]
Số lần lặp: 11
Sai số: 0.000682

Hệ 3 — Jacobi v1
Nghiệm: [-1.000120; -0.000134; 0.999857; 1.999880]
Số lần lặp: 10
Sai số: 0.000470

Hệ 3 — Jacobi v2
Nghiệm: [-1.000120; -0.000134; 0.999857; 1.999880]
Số lần lặp: 10
Sai số: 0.000470

Hệ 4 — Jacobi v1
Nghiệm: [1.742237; 2.683416; -5.037763; -0.100563]
Số lần lặp: 4
Sai số: 0.000065

Hệ 4 — Jacobi v2
Nghiệm: [1.742237; 2.683416; -5.037763; -0.100563]
Số lần lặp: 4
Sai số: 0.000065

